# Carregando o Modelo

Vamos iniciar o processo carregando o modelo treinado em `../model/model-baseline.ipynb` utilizando os dados gerados em `../model/thompson_sampling_contextual_model.json`

In [9]:
import os
import json
import pandas as pd

if os.path.exists('../synthetic_enrichment/offer_catalog.csv'):
    print("📦 Arquivo de ofertas encontrado! Carregando ofertas disponíveis...")
    df = pd.read_csv('../synthetic_enrichment/offer_catalog.csv')
    ofertas_disponiveis = df['category'].unique()
    print("📦 Ofertas disponíveis:", ofertas_disponiveis)
else:
    print("📦 Arquivo de ofertas não encontrado!")

arquivo = '../model/thompson_sampling_contextual_model.json'
if os.path.exists(arquivo):
    print("🧠 Arquivo encontrado! Carregando aprendizado anterior...")
    with open(arquivo, 'r') as f:
        modelos_ts = json.load(f)
else:
    print("🧠 Arquivo não encontrado! Verifique o caminho do arquivo.")

📦 Arquivo de ofertas encontrado! Carregando ofertas disponíveis...
📦 Ofertas disponíveis: <StringArray>
['Serviços', 'Investimento', 'Seguros', 'Crédito']
Length: 4, dtype: str
🧠 Arquivo encontrado! Carregando aprendizado anterior...


### Listando os contextos disponíveis no arquivo.

In [6]:
# Formato Esperado:
print("Formato Esperado: Canal -> Cluster(SocioEconomico) -> Emprestimo -> Grupo da Idade")  

for contexto in modelos_ts.keys():
    print(f"Contexto: {contexto}")

Formato Esperado: Canal -> Cluster(SocioEconomico) -> Emprestimo -> Grupo da Idade
Contexto: Canal_App Push_Cluster_4_Emprestimo_0_age_Adulto
Contexto: Canal_App Push_Cluster_4_Emprestimo_0_age_Jovem
Contexto: Canal_SMS_Cluster_3_Emprestimo_0_age_Adulto
Contexto: Canal_Banner_Cluster_2_Emprestimo_0_age_Senior
Contexto: Canal_Email_Cluster_4_Emprestimo_0_age_Adulto
Contexto: Canal_Banner_Cluster_4_Emprestimo_0_age_Adulto
Contexto: Canal_Email_Cluster_2_Emprestimo_0_age_Adulto
Contexto: Canal_Email_Cluster_3_Emprestimo_2_age_Senior
Contexto: Canal_Email_Cluster_5_Emprestimo_0_age_Adulto
Contexto: Canal_App Push_Cluster_1_Emprestimo_0_age_Jovem
Contexto: Canal_App Push_Cluster_5_Emprestimo_0_age_Jovem
Contexto: Canal_Banner_Cluster_5_Emprestimo_0_age_Jovem
Contexto: Canal_SMS_Cluster_1_Emprestimo_2_age_Jovem
Contexto: Canal_App Push_Cluster_2_Emprestimo_0_age_Adulto
Contexto: Canal_Email_Cluster_2_Emprestimo_0_age_Senior
Contexto: Canal_SMS_Cluster_2_Emprestimo_0_age_Adulto
Contexto: Cana

In [118]:
# calcula confiança por (contexto, oferta) e exibe top 10 globais
rows = []
for contexto, segmento in modelos_ts.items():
    if not isinstance(segmento, dict):
        continue

    alphas_segmento = segmento.get('alphas')
    betas_segmento = segmento.get('betas')

    if not isinstance(alphas_segmento, dict) or not isinstance(betas_segmento, dict):
        continue

    for oferta in ofertas_disponiveis:
        if oferta not in alphas_segmento or oferta not in betas_segmento:
            continue

        confianca = alphas_segmento[oferta] / (alphas_segmento[oferta] + betas_segmento[oferta])
        rows.append({
            'contexto': contexto,
            'oferta': oferta,
            'confianca': confianca
        })

df_confiancas = pd.DataFrame(rows)
df_top10 = df_confiancas.sort_values('confianca', ascending=False).head(10).reset_index(drop=True)
df_top10['confianca'] = df_top10['confianca'].round(4)
print(df_top10)

                                           contexto        oferta  confianca
0       Canal_SMS_Cluster_5_Emprestimo_1_age_Adulto       Seguros     0.8750
1        Canal_SMS_Cluster_2_Emprestimo_0_age_Jovem       Crédito     0.8571
2     Canal_Email_Cluster_0_Emprestimo_0_age_Senior      Serviços     0.8333
3    Canal_Banner_Cluster_3_Emprestimo_1_age_Senior      Serviços     0.8333
4  Canal_App Push_Cluster_3_Emprestimo_1_age_Senior       Seguros     0.8333
5       Canal_SMS_Cluster_3_Emprestimo_1_age_Senior       Crédito     0.7931
6      Canal_Email_Cluster_0_Emprestimo_1_age_Jovem      Serviços     0.7500
7     Canal_Banner_Cluster_0_Emprestimo_1_age_Jovem  Investimento     0.7368
8    Canal_Banner_Cluster_4_Emprestimo_1_age_Senior  Investimento     0.7143
9     Canal_Email_Cluster_3_Emprestimo_1_age_Senior      Serviços     0.6667


In [126]:
import numpy as np

# 0. Fazendo um hack aqui e vou testar o modelo com um Golden Set de 5 perfis representativos
# mas onde ele tem mais confiança.

# 1  Canal_SMS_Cluster_5_Emprestimo_1_age_Adulto       Seguros     0.8750
# 2  Canal_SMS_Cluster_2_Emprestimo_0_age_Jovem        Crédito     0.8571
# 3  Canal_Email_Cluster_0_Emprestimo_0_age_Senior     Serviços    0.8333
# 4  Canal_Banner_Cluster_3_Emprestimo_1_age_Senior    Serviços    0.8333
# 5  Canal_App Push_Cluster_3_Emprestimo_1_age_Senior  Seguros     0.8333

# 1. Criação do Golden Set (5 Perfis Representativos)
golden_set = [
    {
        "cliente": "Cliente A (Adulto Com emprestimo classico.)", 
        "context_segment": "Canal_SMS_Cluster_5_Emprestimo_1_age_Adulto", # Ex: Cluster de alta renda sem empréstimo
        "expectativa_de_negocio": "Seguros"
    },
    {
        "cliente": "Cliente B (Jovem querendo emprestimo)", 
        "context_segment": "Canal_SMS_Cluster_2_Emprestimo_0_age_Jovem", # Ex: Cluster de baixa renda com empréstimo ativo
        "expectativa_de_negocio": "Crédito"
    },
    {
        "cliente": "Cliente C (Senior com tempo livre.)", 
        "context_segment": "Canal_Email_Cluster_0_Emprestimo_0_age_Senior", 
        "expectativa_de_negocio": "Serviços"
    },
    {
        "cliente": "Cliente D (Velho com divida no celular.)", 
        "context_segment": "Canal_Banner_Cluster_3_Emprestimo_1_age_Senior", 
        "expectativa_de_negocio": "Serviços"
    },
    {
        "cliente": "Cliente E (Senior com divida no celular.)", 
        "context_segment": "Canal_App Push_Cluster_3_Emprestimo_1_age_Senior", 
        "expectativa_de_negocio": "Seguros"
    }
]

print("--- AVALIAÇÃO DO GOLDEN SET (TESTE DE SANIDADE) ---\n")

# 2. Avaliação das Decisões do Modelo
for caso in golden_set:
    contexto = caso["context_segment"]
    
    # Verifica se o contexto existe no modelo treinado
    if contexto in modelos_ts:
        # Puxa o "cérebro" treinado daquele segmento específico
        alphas_segmento = modelos_ts[contexto]['alphas']
        betas_segmento = modelos_ts[contexto]['betas']
        
        # O modelo toma a decisão baseada no que aprendeu (Fase de Explotação)
        sampled_theta = {
            offer: np.random.beta(alphas_segmento[offer], betas_segmento[offer]) 
            for offer in ofertas_disponiveis
        }
        oferta_recomendada = max(sampled_theta, key=sampled_theta.get)
        
        # O nível de confiança (taxa de conversão esperada pelo modelo para essa oferta)
        confianca = alphas_segmento[oferta_recomendada] / (alphas_segmento[oferta_recomendada] + betas_segmento[oferta_recomendada])
        
    else:
        oferta_recomendada = "Contexto não encontrado no treino"
        confianca = 0.0

    # 3. Imprime o resultado para auditoria
    print(f"👤 {caso['cliente']} (Contexto: {contexto})")
    print(f"🎯 Oferta Recomendada: {oferta_recomendada} (Confiança do modelo: {confianca:.1%})")
    print(f"🧠 Expectativa de Negócio: {caso['expectativa_de_negocio']}")
    print(f"❓ Resultado: { '✅' if caso['expectativa_de_negocio'] == oferta_recomendada else '❌' }")
    print("-" * 60)

--- AVALIAÇÃO DO GOLDEN SET (TESTE DE SANIDADE) ---

👤 Cliente A (Adulto Com emprestimo classico.) (Contexto: Canal_SMS_Cluster_5_Emprestimo_1_age_Adulto)
🎯 Oferta Recomendada: Seguros (Confiança do modelo: 87.5%)
🧠 Expectativa de Negócio: Seguros
❓ Resultado: ✅
------------------------------------------------------------
👤 Cliente B (Jovem querendo emprestimo) (Contexto: Canal_SMS_Cluster_2_Emprestimo_0_age_Jovem)
🎯 Oferta Recomendada: Seguros (Confiança do modelo: 50.0%)
🧠 Expectativa de Negócio: Crédito
❓ Resultado: ❌
------------------------------------------------------------
👤 Cliente C (Senior com tempo livre.) (Contexto: Canal_Email_Cluster_0_Emprestimo_0_age_Senior)
🎯 Oferta Recomendada: Serviços (Confiança do modelo: 83.3%)
🧠 Expectativa de Negócio: Serviços
❓ Resultado: ✅
------------------------------------------------------------
👤 Cliente D (Velho com divida no celular.) (Contexto: Canal_Banner_Cluster_3_Emprestimo_1_age_Senior)
🎯 Oferta Recomendada: Serviços (Confiança d

## Avaliando o resultado de uma execução.

--- AVALIAÇÃO DO GOLDEN SET (TESTE DE SANIDADE) ---

```
👤 Cliente A (Adulto Com emprestimo classico.) (Contexto: Canal_SMS_Cluster_5_Emprestimo_1_age_Adulto)
🎯 Oferta Recomendada: Seguros (Confiança do modelo: 87.5%)
🧠 Expectativa de Negócio: Seguros
❓ Resultado: ✅
------------------------------------------------------------
👤 Cliente B (Jovem querendo emprestimo) (Contexto: Canal_SMS_Cluster_2_Emprestimo_0_age_Jovem)
🎯 Oferta Recomendada: Seguros (Confiança do modelo: 50.0%)
🧠 Expectativa de Negócio: Crédito
❓ Resultado: ❌ 👈️ A expectativa era fornecer crédito ao Jovem, o que acontece na maioria das vezes mas tem momentos como esse onde falha a expectativa.
------------------------------------------------------------
👤 Cliente C (Senior com tempo livre.) (Contexto: Canal_Email_Cluster_0_Emprestimo_0_age_Senior)
🎯 Oferta Recomendada: Serviços (Confiança do modelo: 83.3%)
🧠 Expectativa de Negócio: Serviços
❓ Resultado: ✅
------------------------------------------------------------
👤 Cliente D (Velho com divida no celular.) (Contexto: Canal_Banner_Cluster_3_Emprestimo_1_age_Senior)
🎯 Oferta Recomendada: Serviços (Confiança do modelo: 83.3%)
🧠 Expectativa de Negócio: Serviços
❓ Resultado: ✅
------------------------------------------------------------
👤 Cliente E (Senior com divida no celular.) (Contexto: Canal_App Push_Cluster_3_Emprestimo_1_age_Senior)
🎯 Oferta Recomendada: Seguros (Confiança do modelo: 83.3%)
🧠 Expectativa de Negócio: Seguros
❓ Resultado: ✅
------------------------------------------------------------
```

### Seguindo de perto as expectativas dos modelos tudo funciona de acordo com o esperado.

### Porém como foram gerados os dados sintéticamente, só poderiamos confiar mais no modelo ao continuar utilizar e fornecer as recompensas 
### reais para o modelo.
